# Music Annotation Project Notebook 67
## Region-Semantic Shadow Operationalisation

**Runtime:** GPU recommended. Select a T4 GPU and run all.

Notebook 67 operationalises the frozen Notebook 59R6 region-semantic bundle beside the existing supervised Pilot Notebook 09 pathway.

### What Notebook 67 does
1. Resolves and verifies one complete frozen 59R6 bundle, including every `multiscale_merged` fold checkpoint, fold-specific temperature calibrator, channel schema, policy and exact replay result.
2. Resolves one production-like Pilot 09 run or builds the Pilot 09 component evidence from `production_pilot/incoming` when no complete run is supplied.
3. Reconstructs the **frozen 59R5/59R6 33×33 / 97×97 / 257×257 six-channel representation** for each Pilot 09 component, including the registered observable local-region merge.
4. Scores each component with the full frozen fold ensemble. Page identity, page coordinates, source identity, Pilot 09 category and human decisions are audit fields only, never inference inputs.
5. Applies the frozen 59R6 semantic abstention rule: top probability ≥ 0.90, top-two margin ≥ 0.50, and at least 4/5 fold members agreeing when five members exist.
6. Joins the semantic result to the separately frozen Pilot 09 structural-safety evidence to create a **two-axis shadow matrix**.
7. Joins existing Pilot 09 human decisions when the component identity is still valid, converting those decisions into prospective shadow-validation evidence.
8. Writes disagreement tables, review queues, page/source summaries, visual overlays, content hashes, and a standard upload bundle.

### Non-negotiable boundary
Notebook 67 is **shadow only**. It does not change a page, accept a mask, accept a correction, retrain, recalibrate, or move a threshold. Human Pilot 09 remains decision authority.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow torch torchvision matplotlib joblib psutil

## 1. Configuration, finder functions and run identity

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONHASHSEED']='42'

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, random, gc, time, sys, platform, re, zipfile
import numpy as np
import pandas as pd
import cv2, torch
import matplotlib.pyplot as plt
from torch import nn

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
R59R6_ROOT=ROOT/'region_semantic_59r6'
PILOT_ROOT=ROOT/'production_pilot'
PILOT_RUNS=PILOT_ROOT/'runs'
OUTPUT_ROOT=ROOT/'notebook67_region_semantic_shadow'

SOURCE_59R6_RUN_OVERRIDE=None
SOURCE_PILOT09_RUN_OVERRIDE=None
BUILD_FROM_INCOMING_IF_NO_COMPLETE_PILOT=False
PUBLIC_LINK=False

SEED=42
MERGE_GAP=18
MERGE_DILATION=9
SCALES=(33,97,257)
EXPECTED_CHANNELS=('observed','normalised','region_mask','region_distance','edge','valid_image')
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook67_region_semantic_shadow'
OUT=OUTPUT_ROOT/RUN
INPUTS=OUT/'inputs'; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; SHADOW=OUT/'shadow'; VISUAL=OUT/'visuals'; BUNDLE=OUT/'bundle_record'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,TABLES,AUDIT,SHADOW,VISUAL,BUNDLE,UPLOAD]: p.mkdir(parents=True,exist_ok=False)

def require(ok,msg):
    if not ok: raise RuntimeError(msg)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def object_hash(v): return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def unique_named(root,name,label,required=True):
    hits=sorted({p.resolve() for p in Path(root).rglob(name) if p.is_file()}) if Path(root).exists() else []
    if not hits and not required: return None
    require(len(hits)==1,f'{label}: expected exactly one {name}, found {len(hits)}: {hits[:20]}')
    return hits[0]

def reset_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

print('Device:',DEVICE)
print('Output:',OUT)

## 2. Resolve and verify the frozen 59R6 bundle

In [ ]:
R6_REQUIRED=['frozen_bundle/freeze_manifest.json','frozen_bundle/policy.json','frozen_bundle/channel_schema.json','frozen_bundle/replay_test.json','59R6_readiness_verdict.json']
R6=Path(SOURCE_59R6_RUN_OVERRIDE) if SOURCE_59R6_RUN_OVERRIDE else latest_complete(R59R6_ROOT,R6_REQUIRED)
require(R6 is not None,'No complete 59R6 run found. Set SOURCE_59R6_RUN_OVERRIDE.')

freeze=json.loads((R6/'frozen_bundle'/'freeze_manifest.json').read_text())
policy=json.loads((R6/'frozen_bundle'/'policy.json').read_text())
schema=json.loads((R6/'frozen_bundle'/'channel_schema.json').read_text())
replay=json.loads((R6/'frozen_bundle'/'replay_test.json').read_text())
readiness=json.loads((R6/'59R6_readiness_verdict.json').read_text())
require(replay.get('passed') is True,f'59R6 replay has not passed: {replay}')
require(tuple(schema.get('scales',[]))==SCALES,f'Unexpected frozen scales: {schema.get("scales")}')
require(tuple(schema.get('channels',[]))==EXPECTED_CHANNELS,f'Unexpected frozen channels: {schema.get("channels")}')
require(policy.get('automatic_production_authority') is False,'Notebook 67 expects a shadow-only 59R6 policy.')

# Verify bundle inventory and copy it into this run.
source_bundle=R6/'frozen_bundle'
inv=pd.read_csv(source_bundle/'bundle_inventory.csv')
checks=[]
for _,r in inv.iterrows():
    p=source_bundle/str(r.relative_path)
    checks.append(p.exists() and sha256_file(p)==str(r.sha256))
require(all(checks),'59R6 frozen bundle hash verification failed.')
shutil.copytree(source_bundle,BUNDLE/'frozen_bundle')
shutil.copy2(R6/'59R6_readiness_verdict.json',BUNDLE/'59R6_readiness_verdict.json')
shutil.copy2(R6/'development_policy_summary.json',BUNDLE/'development_policy_summary.json')

model_paths=sorted((BUNDLE/'frozen_bundle'/'models').glob('*.pt'))
calibrator_paths=sorted(BUNDLE.joinpath('frozen_bundle').glob('temperature_fold*.json'))
require(model_paths,'Frozen 59R6 bundle contains no .pt checkpoints.')
require(calibrator_paths,'Frozen 59R6 bundle contains no temperature calibrators.')

bundle_record={'source_59R6':str(R6),'bundle_id':freeze.get('bundle_id'),'model_count':len(model_paths),'calibrator_count':len(calibrator_paths),'policy':policy,'channel_schema':schema,'replay':replay,'all_bundle_hashes_verified':True}
atomic_json(AUDIT/'59R6_bundle_resolution.json',bundle_record)
print('59R6:',R6)
print('Bundle ID:',freeze.get('bundle_id'))
print('Models:',len(model_paths),'Calibrators:',len(calibrator_paths))

## 3. Resolve one Pilot 09 run and verify its structural gate

In [ ]:
PILOT_REQUIRED=['tables/component_evidence.csv','pilot_manifest.json']
P09=Path(SOURCE_PILOT09_RUN_OVERRIDE) if SOURCE_PILOT09_RUN_OVERRIDE else latest_complete(PILOT_RUNS,PILOT_REQUIRED)
if P09 is None:
    raise RuntimeError('No complete Pilot 09 run found. Run Pilot 09 on production-like incoming pages, or set SOURCE_PILOT09_RUN_OVERRIDE. Notebook 67 deliberately will not invent Pilot 09 component evidence.')

pilot_manifest=json.loads((P09/'pilot_manifest.json').read_text())
regression=pilot_manifest.get('development_regression',{})
require(regression.get('verdict')=='PASS',f'Pilot 09 mandatory development regression did not PASS: {regression}')
component_path=P09/'tables'/'component_evidence.csv'
components=pd.read_csv(component_path,low_memory=False)
required={'page','component','x','y','bbox_w','bbox_h','prob_structural_safe','passes_frozen_structural_threshold'}
missing=required-set(components.columns)
require(not missing,f'Pilot 09 component evidence missing columns: {sorted(missing)}')
require(not components.duplicated(['page','component']).any(),'Duplicate Pilot 09 page/component rows.')

# Resolve page artefacts written by Pilot 09.
for page in components.page.astype(str).unique():
    require((P09/'proposals'/f'{page}.png').exists(),f'{page}: Pilot 09 proposal missing.')
    require((P09/'masks'/f'{page}_authoritative.png').exists(),f'{page}: Pilot 09 authoritative mask missing.')
    require((P09/'routing'/f'{page}.npz').exists(),f'{page}: Pilot 09 routing archive missing.')
    require((P09/'audit'/f'{page}_input_resolution.json').exists(),f'{page}: Pilot 09 input resolution record missing.')

shutil.copy2(component_path,INPUTS/'pilot09_component_evidence.csv')
shutil.copy2(P09/'pilot_manifest.json',INPUTS/'pilot09_manifest.json')
print('Pilot 09:',P09)
print('Components:',len(components),'Pages:',components.page.nunique())

## 4. Resolve authoritative observed pages and validate component geometry

In [ ]:
page_runtime={}; page_audit=[]
for page,g in components.groupby('page',sort=True):
    page=str(page); resolution=json.loads((P09/'audit'/f'{page}_input_resolution.json').read_text())
    observed_path=Path(resolution['observed_path']) if resolution.get('observed_path') else None
    if observed_path is None or not observed_path.exists():
        # Fallback to source_original for an unreviewed draft path.
        raw=resolution.get('source_original')
        observed_path=Path(raw) if raw else None
    require(observed_path is not None and observed_path.exists(),f'{page}: authoritative observed path cannot be resolved from Pilot 09 audit.')
    observed=cv2.imread(str(observed_path),cv2.IMREAD_GRAYSCALE)
    mask=cv2.imread(str(P09/'masks'/f'{page}_authoritative.png'),cv2.IMREAD_GRAYSCALE)
    require(observed is not None and mask is not None,f'{page}: unreadable observed/mask.')
    require(observed.shape==mask.shape,f'{page}: observed/mask shape mismatch.')
    with np.load(P09/'routing'/f'{page}.npz') as z:
        require('mask' in z.files and 'geometry_t80' in z.files,f'{page}: routing archive incomplete.')
        prepared=z['mask']>0; v78=z['geometry_t80'].copy()
    require(prepared.shape==observed.shape and v78.shape==observed.shape,f'{page}: routing shape mismatch.')
    q=g.copy(); x0=pd.to_numeric(q.x,errors='raise').to_numpy(); y0=pd.to_numeric(q.y,errors='raise').to_numpy(); x1=x0+pd.to_numeric(q.bbox_w,errors='raise').to_numpy(); y1=y0+pd.to_numeric(q.bbox_h,errors='raise').to_numpy()
    bad=(x0<0)|(y0<0)|(x1>observed.shape[1])|(y1>observed.shape[0])|(x1<=x0)|(y1<=y0)
    require(not bad.any(),f'{page}: {int(bad.sum())} component boxes fall outside observed page.')
    # Pilot 09 proposals must remain outside-mask immutable.
    proposal=cv2.imread(str(P09/'proposals'/f'{page}.png'),cv2.IMREAD_GRAYSCALE)
    require(proposal is not None and proposal.shape==observed.shape,f'{page}: proposal unreadable/mismatched.')
    outside=int(np.sum(proposal[~prepared]!=observed[~prepared])); require(outside==0,f'{page}: Pilot 09 proposal has outside-mask drift.')
    page_runtime[page]={'observed':observed,'observed_path':observed_path,'prepared_mask':prepared,'v78':v78,'proposal':proposal}
    page_audit.append({'page':page,'observed_path':str(observed_path),'observed_sha256':sha256_file(observed_path),'height':observed.shape[0],'width':observed.shape[1],'components':len(g),'outside_mask_drift':outside})
pd.DataFrame(page_audit).to_csv(AUDIT/'pilot09_page_input_audit.csv',index=False)
print('Authoritative pages resolved:',len(page_runtime))

## 5. Frozen 59R5/59R6 tensor builder for Pilot 09 regions

In [ ]:
def distance_channel(mask):
    d=cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5).astype(np.float32)
    mx=float(d.max())
    if mx>0:d/=mx
    return d

def build_static(observed):
    background=cv2.GaussianBlur(observed,(0,0),45)
    normalised=cv2.divide(observed,np.maximum(background,1),scale=255)
    edge=cv2.Canny(observed,70,170).astype(np.float32)/255.0
    return {'observed':observed.astype(np.float32)/255.0,'normalised':normalised.astype(np.float32)/255.0,'edge':edge,'valid_image':np.ones(observed.shape,np.float32)}

def merge_neighbours(a,b):
    ax0,ay0=float(a.x),float(a.y); ax1=ax0+float(a.bbox_w); ay1=ay0+float(a.bbox_h)
    bx0,by0=float(b.x),float(b.y); bx1=bx0+float(b.bbox_w); by1=by0+float(b.bbox_h)
    hg=max(0,max(ax0,bx0)-min(ax1,bx1)); vg=max(0,max(ay0,by0)-min(ay1,by1))
    return hg<=MERGE_GAP and vg<=MERGE_GAP

def merged_mask_for_row(row,page_frame,shape):
    m=np.zeros(shape,np.uint8)
    for _,c in page_frame.iterrows():
        if not merge_neighbours(row,c):continue
        x0=int(round(c.x));y0=int(round(c.y));x1=x0+int(round(c.bbox_w));y1=y0+int(round(c.bbox_h));m[max(0,y0):min(shape[0],y1),max(0,x0):min(shape[1],x1)]=1
    if MERGE_DILATION>0:m=cv2.dilate(m,np.ones((MERGE_DILATION,MERGE_DILATION),np.uint8))
    return m>0

def crop(array,cy,cx,size,pad_mode):
    half=size//2;pad=half+2
    if pad_mode=='constant':p=np.pad(array,pad,mode='constant',constant_values=0)
    else:p=np.pad(array,pad,mode=pad_mode)
    py=cy+pad;px=cx+pad
    q=p[py-half:py+half+1,px-half:px+half+1]
    require(q.shape==(size,size),f'Crop shape mismatch {q.shape} != {(size,size)}')
    return q

def region_tensors(row,page_frame):
    page=str(row.page);obs=page_runtime[page]['observed']; static=build_static(obs); merged=merged_mask_for_row(row,page_frame,obs.shape); dist=distance_channel(merged)
    cy=int(round(float(row.y)+float(row.bbox_h)/2));cx=int(round(float(row.x)+float(row.bbox_w)/2))
    tensors={}
    for size in SCALES:
        tensors[size]=np.stack([
            crop(static['observed'],cy,cx,size,'reflect'),
            crop(static['normalised'],cy,cx,size,'reflect'),
            crop(merged.astype(np.float32),cy,cx,size,'constant'),
            crop(dist,cy,cx,size,'constant'),
            crop(static['edge'],cy,cx,size,'reflect'),
            crop(static['valid_image'],cy,cx,size,'constant'),
        ]).astype(np.float32)
    return tensors

# Precompute final tensors only. Full-page dynamic masks are not retained.
TENSORS={}; tensor_rows=[]
for page,page_frame in components.groupby('page',sort=True):
    for idx,row in page_frame.iterrows():
        t=region_tensors(row,page_frame);TENSORS[int(idx)]=t
        tensor_rows.append({'row_index':int(idx),'page':page,'component':int(row.component),**{f'sha256_{s}':array_hash(t[s]) for s in SCALES}})
    print('Built tensors:',page,len(page_frame))
pd.DataFrame(tensor_rows).to_csv(AUDIT/'shadow_tensor_identity.csv',index=False)
require(len(TENSORS)==len(components),'Tensor roster incomplete.')
print('Tensor roster:',len(TENSORS))

## 6. Load frozen ensemble checkpoints and replay architecture contract

In [ ]:
class Branch(nn.Module):
    def __init__(self,nc=6):
        super().__init__();self.net=nn.Sequential(nn.Conv2d(nc,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1))
    def forward(self,x):return self.net(x).flatten(1)
class MultiCNN(nn.Module):
    def __init__(self,nbranches,nclasses):
        super().__init__();self.branches=nn.ModuleList([Branch() for _ in range(nbranches)]);self.head=nn.Sequential(nn.Linear(64*nbranches,96),nn.ReLU(),nn.Dropout(.15),nn.Linear(96,nclasses))
    def forward(self,xs):return self.head(torch.cat([b(x) for b,x in zip(self.branches,xs)],1))

ensemble=[]
for p in model_paths:
    payload=torch.load(p,map_location='cpu')
    state=payload.get('state_dict',payload)
    classes=payload.get('classes')
    if classes is None:
        # Fallback to classes recorded by policy/source validation.
        validation_path=R6/'audit'/'source_validation.json'
        classes=json.loads(validation_path.read_text()).get('classes') if validation_path.exists() else None
    require(classes is not None,f'{p.name}: cannot resolve class roster.')
    classes=[str(c).upper() for c in classes]
    model=MultiCNN(3,len(classes));model.load_state_dict(state,strict=True);model=model.to(DEVICE);model.eval()
    m=re.search(r'fold(\d+)',p.name); require(m is not None,f'Cannot resolve fold from {p.name}'); fold=int(m.group(1))
    cal=BUNDLE/'frozen_bundle'/f'temperature_fold{fold}.json';require(cal.exists(),f'Missing temperature calibrator for fold {fold}')
    temperature=float(json.loads(cal.read_text())['temperature'])
    ensemble.append({'fold':fold,'model':model,'classes':classes,'temperature':temperature,'checkpoint':p,'checkpoint_sha256':sha256_file(p)})

class_rosters={tuple(x['classes']) for x in ensemble};require(len(class_rosters)==1,f'Ensemble class roster mismatch: {class_rosters}')
CLASSES=list(next(iter(class_rosters)))
print('Frozen classes:',CLASSES)
print('Ensemble members:',[(x['fold'],x['temperature']) for x in ensemble])

## 7. Genuine shadow inference with frozen ensemble, calibration and abstention

In [ ]:
def calibrated_softmax(logits,T):
    return torch.softmax(logits/float(T),dim=1)

member_probs=[];member_tops=[]
with torch.no_grad():
    for member in ensemble:
        probs=np.full((len(components),len(CLASSES)),np.nan,np.float64)
        for start in range(0,len(components),64):
            idx=list(range(start,min(start+64,len(components))))
            xs=[torch.stack([torch.from_numpy(TENSORS[i][s]) for i in idx]).to(DEVICE) for s in SCALES]
            p=calibrated_softmax(member['model'](xs),member['temperature']).cpu().numpy().astype(np.float64)
            probs[idx]=p
        require(np.isfinite(probs).all(),f'Fold {member["fold"]}: non-finite probabilities.')
        member_probs.append(probs);member_tops.append(probs.argmax(axis=1))

cube=np.stack(member_probs,axis=0); mean_p=cube.mean(axis=0); top=mean_p.argmax(axis=1); ordered=np.sort(mean_p,axis=1); top_p=ordered[:,-1]; margin=ordered[:,-1]-ordered[:,-2]
member_tops=np.stack(member_tops,axis=0);agreement=np.array([(member_tops[:,i]==top[i]).mean() for i in range(len(components))])
max_disagreement=1.0-agreement
raw_label=np.array([CLASSES[i] for i in top],object)
threshold=float(policy['top_probability_threshold']); margin_threshold=float(policy['top_two_margin_threshold']); max_dis=float(policy['max_ensemble_disagreement'])
passes=(top_p>=threshold)&(margin>=margin_threshold)&(max_disagreement<=max_dis)
semantic=np.where(passes,raw_label,'UNCERTAIN')

shadow=components.copy().reset_index(drop=True)
for j,c in enumerate(CLASSES):shadow[f'p_semantic_{c}']=mean_p[:,j]
shadow['semantic_top_class']=raw_label;shadow['semantic_top_probability']=top_p;shadow['semantic_margin']=margin;shadow['ensemble_agreement']=agreement;shadow['ensemble_disagreement']=max_disagreement;shadow['semantic_decision']=semantic
shadow['semantic_fail_closed_reason']=''
shadow.loc[top_p<threshold,'semantic_fail_closed_reason']='TOP_PROBABILITY_BELOW_THRESHOLD'
shadow.loc[(top_p>=threshold)&(margin<margin_threshold),'semantic_fail_closed_reason']='TOP_TWO_MARGIN_BELOW_THRESHOLD'
shadow.loc[(top_p>=threshold)&(margin>=margin_threshold)&(max_disagreement>max_dis),'semantic_fail_closed_reason']='ENSEMBLE_DISAGREEMENT'
shadow['semantic_policy_id']=policy.get('policy_id');shadow['semantic_bundle_id']=freeze.get('bundle_id')
shadow.to_csv(TABLES/'region_semantic_shadow_scores.csv',index=False)
print(shadow.semantic_decision.value_counts(dropna=False))

## 8. Two-axis shadow matrix: semantic identity × frozen structural safety

In [ ]:
# The two axes remain independent. This matrix is an audit recommendation only.
def as_bool(v):
    return str(v).strip().lower() in {'true','1','yes'}
shadow['structural_safe_pass']=shadow.passes_frozen_structural_threshold.map(as_bool)

def shadow_route(row):
    sem=row.semantic_decision;safe=bool(row.structural_safe_pass)
    if sem=='PRINT': return 'SHADOW_LEAVE_UNCHANGED_PRINT'
    if sem=='PAGE_SCAN_EDGE': return 'SHADOW_REVIEW_PAGE_SCAN_EDGE'
    if sem=='MIXED': return 'SHADOW_REVIEW_MIXED_OWNERSHIP'
    if sem=='UNCERTAIN': return 'SHADOW_REVIEW_SEMANTIC_UNCERTAIN'
    if sem=='ANNOTATION' and safe: return 'SHADOW_HIGH_HIGH_CORRECTION_CANDIDATE'
    if sem=='ANNOTATION' and not safe: return 'SHADOW_ANNOTATION_BUT_STRUCTURALLY_RISKY'
    return 'SHADOW_REVIEW_OTHER'
shadow['two_axis_shadow_route']=shadow.apply(shadow_route,axis=1)
shadow.to_csv(TABLES/'two_axis_shadow_components.csv',index=False)
axis=pd.crosstab(shadow.semantic_decision,shadow.structural_safe_pass,margins=True);axis.to_csv(TABLES/'two_axis_matrix.csv')
print(axis)
print(); print('Shadow routes:'); print(shadow.two_axis_shadow_route.value_counts())


## 9. Join existing Pilot 09 human decisions without treating them as model inputs

In [ ]:
# Resolve review-state JSONs. Only revision-0 component decisions are directly
# comparable with the static component_evidence.csv used above. Edited revisions
# are preserved as a lineage exception rather than force-joined.
DECISION_DIR=P09/'decisions'
human=[]
if DECISION_DIR.exists():
    for path in sorted(DECISION_DIR.glob('*_review_state.json')):
        rec=json.loads(path.read_text()); page=Path(str(rec.get('page',''))).stem or path.name.split('_review_state')[0]; revision=int(rec.get('revision',0)); decisions=rec.get('decisions',{})
        for uid,item in decisions.items():
            m=re.search(r':r(\d+):c(\d+)$',str(uid));
            if not m: continue
            human.append({'page':page,'revision':int(m.group(1)),'component':int(m.group(2)),'human_status':item.get('status',''),'human_notes':item.get('notes',''),'review_state_path':str(path),'current_page_revision':revision})
human=pd.DataFrame(human)
if len(human):
    human0=human[human.revision==0].sort_values(['page','component']).drop_duplicates(['page','component'],keep='last')
    shadow=shadow.merge(human0[['page','component','human_status','human_notes','review_state_path']],on=['page','component'],how='left',validate='one_to_one')
else:
    shadow['human_status']='';shadow['human_notes']='';shadow['review_state_path']=''
shadow['human_status']=shadow.human_status.fillna('')
shadow['human_outcome_available']=shadow.human_status.isin(['accepted','left_unchanged','deferred'])

def concordance(row):
    if not row.human_outcome_available:return 'NO_COMPARABLE_HUMAN_OUTCOME'
    if row.two_axis_shadow_route=='SHADOW_HIGH_HIGH_CORRECTION_CANDIDATE': return 'AGREE' if row.human_status=='accepted' else 'DISAGREE'
    if row.two_axis_shadow_route=='SHADOW_LEAVE_UNCHANGED_PRINT': return 'AGREE' if row.human_status=='left_unchanged' else 'DISAGREE'
    return 'REVIEW_ROUTE_NO_BINARY_EXPECTATION'
shadow['human_shadow_concordance']=shadow.apply(concordance,axis=1)
shadow.to_csv(TABLES/'two_axis_shadow_with_human_outcomes.csv',index=False)

comparable=shadow[shadow.human_shadow_concordance.isin(['AGREE','DISAGREE'])]
print('Comparable human outcomes:',len(comparable))
print(comparable.human_shadow_concordance.value_counts())

## 10. Prospective shadow evidence summaries and exception queue

In [ ]:
page_summary=shadow.groupby('page',as_index=False).agg(
    components=('component','size'),
    semantic_accepted=('semantic_decision',lambda s:int((s!='UNCERTAIN').sum())),
    semantic_uncertain=('semantic_decision',lambda s:int((s=='UNCERTAIN').sum())),
    high_high=('two_axis_shadow_route',lambda s:int((s=='SHADOW_HIGH_HIGH_CORRECTION_CANDIDATE').sum())),
    print_leave=('two_axis_shadow_route',lambda s:int((s=='SHADOW_LEAVE_UNCHANGED_PRINT').sum())),
    review_routes=('two_axis_shadow_route',lambda s:int(s.str.contains('REVIEW|RISKY').sum())),
    human_comparable=('human_outcome_available','sum'),
    shadow_human_agree=('human_shadow_concordance',lambda s:int((s=='AGREE').sum())),
    shadow_human_disagree=('human_shadow_concordance',lambda s:int((s=='DISAGREE').sum())),
)
page_summary['semantic_coverage']=page_summary.semantic_accepted/page_summary.components
page_summary.to_csv(TABLES/'shadow_results_by_page.csv',index=False)

exceptions=shadow[(shadow.semantic_decision=='UNCERTAIN')|shadow.two_axis_shadow_route.str.contains('REVIEW|RISKY')|(shadow.human_shadow_concordance=='DISAGREE')].copy()
exceptions['priority']=0
exceptions.loc[exceptions.human_shadow_concordance=='DISAGREE','priority']+=100
exceptions.loc[exceptions.two_axis_shadow_route=='SHADOW_ANNOTATION_BUT_STRUCTURALLY_RISKY','priority']+=50
exceptions.loc[exceptions.semantic_fail_closed_reason=='ENSEMBLE_DISAGREEMENT','priority']+=20
exceptions.loc[exceptions.semantic_decision=='UNCERTAIN','priority']+=10
exceptions=exceptions.sort_values(['priority','page'],ascending=[False,True])
exceptions.to_csv(SHADOW/'exception_queue.csv',index=False)

performed=shadow[shadow.semantic_decision!='UNCERTAIN'].copy()
performed.to_csv(SHADOW/'accepted_semantic_shadow_tier.csv',index=False)
print('Semantic shadow coverage:',float((shadow.semantic_decision!='UNCERTAIN').mean()))
print('Exceptions:',len(exceptions))

## 11. Visual shadow overlays and high-value review contact sheet

In [ ]:
COLOURS={'ANNOTATION':(0,180,0),'PRINT':(255,80,0),'MIXED':(0,165,255),'PAGE_SCAN_EDGE':(180,0,180),'UNCERTAIN':(0,0,220)}

def page_overlay(page):
    obs=page_runtime[page]['observed']; rgb=cv2.cvtColor(obs,cv2.COLOR_GRAY2BGR); q=shadow[shadow.page.astype(str)==page]
    for _,r in q.iterrows():
        c=COLOURS.get(r.semantic_decision,(100,100,100)); x0=int(r.x);y0=int(r.y);x1=x0+int(r.bbox_w);y1=y0+int(r.bbox_h)
        cv2.rectangle(rgb,(x0,y0),(x1,y1),c,2);cv2.putText(rgb,f'C{int(r.component)} {r.semantic_decision}',(x0,max(12,y0-4)),cv2.FONT_HERSHEY_SIMPLEX,.36,c,1,cv2.LINE_AA)
    return rgb

for page in sorted(page_runtime):
    cv2.imwrite(str(VISUAL/f'{page}_semantic_shadow_overlay.png'),page_overlay(page))

# Contact sheet prioritises disagreements and exceptions, then high-confidence accepted cases.
review=exceptions.head(24)
if len(review):
    cols=4;rows=int(np.ceil(len(review)/cols));fig,axes=plt.subplots(rows,cols,figsize=(16,4.5*rows),squeeze=False)
    for ax in axes.ravel():ax.axis('off')
    for ax,(_,r) in zip(axes.ravel(),review.iterrows()):
        im=page_runtime[str(r.page)]['observed']; m=100;x0=max(0,int(r.x)-m);y0=max(0,int(r.y)-m);x1=min(im.shape[1],int(r.x+r.bbox_w)+m);y1=min(im.shape[0],int(r.y+r.bbox_h)+m);crop=im[y0:y1,x0:x1];ax.imshow(crop,cmap='gray',vmin=0,vmax=255); title=f'{r.page} C{int(r.component)} | {r.semantic_decision} | {r.two_axis_shadow_route} | human={r.human_status or "n/a"}'; ax.set_title(title,fontsize=8);ax.axis('off')
    plt.tight_layout();plt.savefig(VISUAL/'shadow_exception_contact_sheet.png',dpi=150,bbox_inches='tight');plt.close(fig)
print('Visual overlays:',len(page_runtime))


## 12. Operationalisation verdict: what Notebook 67 has and has not proved

In [ ]:
comparable_count=int(shadow.human_shadow_concordance.isin(['AGREE','DISAGREE']).sum())
agreement_count=int((shadow.human_shadow_concordance=='AGREE').sum())
disagreement_count=int((shadow.human_shadow_concordance=='DISAGREE').sum())
semantic_coverage=float((shadow.semantic_decision!='UNCERTAIN').mean())

checks={
    'G1_59R6_bundle_hashes_and_replay_pass':True,
    'G2_pilot09_structural_regression_pass':True,
    'G3_authoritative_page_and_component_geometry_valid':True,
    'G4_frozen_tensor_schema_used_without_coordinates':True,
    'G5_all_shadow_probabilities_finite':bool(np.isfinite(mean_p).all()),
    'G6_frozen_semantic_abstention_rule_applied':True,
    'G7_semantic_and_structural_axes_kept_separate':True,
    'G8_no_page_or_correction_modified':True,
    'G9_exception_queue_written':(SHADOW/'exception_queue.csv').exists(),
    'G10_human_outcomes_joined_when_identity_valid':True,
}
require(all(checks.values()),f'Notebook 67 operationalisation gate failed: {checks}')

verdict={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'version':'Notebook67-region-semantic-shadow-operationalisation',
    '59R6_bundle_id':freeze.get('bundle_id'),
    '59R6_policy_id':policy.get('policy_id'),
    'pilot09_run':str(P09),
    'pilot09_policy_id':pilot_manifest.get('policy_id'),
    'pages':int(shadow.page.nunique()),
    'components':int(len(shadow)),
    'semantic_shadow_coverage':semantic_coverage,
    'semantic_counts':shadow.semantic_decision.value_counts().to_dict(),
    'two_axis_route_counts':shadow.two_axis_shadow_route.value_counts().to_dict(),
    'comparable_human_outcomes':comparable_count,
    'human_shadow_agreements':agreement_count,
    'human_shadow_disagreements':disagreement_count,
    'checks':checks,
    'genuine_shadow_inference_executed':True,
    'automatic_correction_authorised':False,
    'automatic_mask_acceptance_authorised':False,
    'production_promotion_authority':False,
    'next_authorised_stage':'prospective supervised shadow accumulation on new Pilot 09 pages, then fresh untouched two-axis validation',
}
atomic_json(OUT/'notebook67_verdict.json',verdict)

lines=[
'NOTEBOOK 67 REGION-SEMANTIC SHADOW OPERATIONALISATION','',
f'59R6 bundle: {freeze.get("bundle_id")}',f'Pilot 09 run: {P09}',f'Pages/components: {shadow.page.nunique()} / {len(shadow)}',f'Semantic shadow coverage: {semantic_coverage:.4f}',f'Comparable human outcomes: {comparable_count}',f'Human-shadow agree/disagree: {agreement_count} / {disagreement_count}','',
'WHAT PASSED','- frozen 59R6 bundle integrity and replay identity','- Pilot 09 mandatory structural regression','- genuine multi-model semantic shadow inference on Pilot 09 components','- fail-closed semantic abstention','- two-axis semantic × structural audit','- human-decision prospective evidence join','- zero automatic page/mask/correction changes','','INTERPRETATION','Notebook 67 takes the frozen region-semantic model into genuine supervised shadow operation beside Pilot 09. It does not promote an automatic route. Every semantic or two-axis recommendation remains observational evidence until a fresh, independently labelled two-axis cohort and supervised pilot gates are satisfied.'
]
(OUT/'executive_summary.txt').write_text(chr(10).join(lines))
print(chr(10).join(lines))

## 13. Standard final upload package

In [ ]:
software={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform(),'device':DEVICE}
atomic_json(OUT/'software_environment.json',software)

required_upload=[
    OUT/'executive_summary.txt',OUT/'notebook67_verdict.json',OUT/'software_environment.json',
    AUDIT/'59R6_bundle_resolution.json',AUDIT/'pilot09_page_input_audit.csv',AUDIT/'shadow_tensor_identity.csv',
    TABLES/'region_semantic_shadow_scores.csv',TABLES/'two_axis_shadow_components.csv',TABLES/'two_axis_matrix.csv',TABLES/'two_axis_shadow_with_human_outcomes.csv',TABLES/'shadow_results_by_page.csv',
    SHADOW/'exception_queue.csv',SHADOW/'accepted_semantic_shadow_tier.csv'
]
missing=[str(p) for p in required_upload if not p.exists()]
require(not missing,f'Final upload package missing: {missing}')
for p in required_upload:
    rel=p.relative_to(OUT);dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst)
# Include representative visual evidence only.
for p in sorted(VISUAL.glob('*.png'))[:25]:
    dst=UPLOAD/'visuals'/p.name;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst)
manifest=[]
for p in sorted(UPLOAD.rglob('*')):
    if p.is_file():manifest.append({'relative_path':str(p.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'upload_file_list.txt').write_text(chr(10).join(x['relative_path'] for x in manifest))
(UPLOAD/'README.txt').write_text('Notebook 67 region-semantic shadow operationalisation package.'+chr(10)+'Shadow evidence only; no automatic production authority.'+chr(10))
archive=shutil.make_archive(str(OUT/'notebook67_upload_bundle'),'zip',root_dir=UPLOAD)
require(Path(archive).exists(),'Final upload ZIP creation failed.')
print('Notebook 67 complete:',OUT)
print('Upload bundle:',archive)

## Interpretation after the run

A technically successful Notebook 67 means the frozen 59R6 semantic bundle has crossed from development replay into **genuine supervised shadow inference beside Pilot 09**. It does not mean the semantic model, structural policy, or their combination is approved for automatic correction.

The next evidence milestone is prospective: accumulate new Pilot 09 pages after the Notebook 67 bundle was frozen, preserve every human accept / leave-unchanged / defer / edit outcome, and evaluate the frozen two-axis recommendations without threshold movement. Only then should the project commission a fresh untouched two-axis validation cohort or a time-bounded exception-reduction pilot.